In [61]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping


In [62]:
file_path = "HTRU_2 2.csv"

column_names = [
    "mean_ip",
    "std_ip",
    "skewness_ip",
    "kurtosis_ip",
    "mean_dm_snr",
    "std_dm_snr",
    "skewness_dm_snr",
    "kurtosis_dm_snr",
    "target",
]

df = pd.read_csv(file_path, header=None, names=column_names)

X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
classes = np.unique(y_train)

In [63]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

###Models

In [64]:
# logistic reg
logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, y_train)

# random forest
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)

# XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

# LightGBM
lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42
)
lgbm.fit(X_train, y_train)


[LightGBM] [Info] Number of positive: 1311, number of negative: 13007
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002534 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 14318, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.091563 -> initscore=-2.294697
[LightGBM] [Info] Start training from score -2.294697


LGBMClassifier(n_estimators=300, random_state=42,
               scale_pos_weight=np.float64(9.921434019832189))

In [65]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# MLP
mlp = Sequential([
    Input(shape=(X_train_s.shape[1],)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

mlp.compile(optimizer="adam", loss="binary_crossentropy")

mlp.fit(
    X_train_s,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    class_weight=class_weights,
    verbose=0
)

In [68]:
# evaluation
def eval_model(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC_AUC": roc_auc_score(y_true, y_prob)
    }

results = []
mlp_prob = mlp.predict(X_test_s).ravel()
mlp_pred = (mlp_prob >= 0.5).astype(int)
results.append(eval_model("MLP", y_test, mlp_pred, mlp_prob))

112/112 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [69]:

models = {
    "LogReg": logreg,
    "RF": rf,
    "XGB": xgb,
    "LGBM": lgbm
}

for name, model in models.items():
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    results.append(eval_model(name, y_test, pred, prob))

# results
results_df = pd.DataFrame(results).sort_values(by="ROC_AUC", ascending=False)

print("\nModel Comparison:")
print(results_df.round(4))


Model Comparison:
    Model  Accuracy  Precision  Recall      F1  ROC_AUC
0     MLP    0.9684     0.7807  0.9116  0.8411   0.9774
1  LogReg    0.9693     0.7824  0.9207  0.8459   0.9728
3     XGB    0.9749     0.8381  0.8994  0.8676   0.9717
4    LGBM    0.9779     0.8716  0.8902  0.8808   0.9711
2      RF    0.9804     0.9188  0.8628  0.8899   0.9683
